# CUB-200 Model Training

This standalone notebook trains a multi-class CUB-200 bird classifier using transfer learning.

The workflow is divided into clear stages: importing dependencies, configuring the experiment,
preparing reproducible data loaders, constructing a transfer-learning model, defining the epoch
logic, training with validation and overfitting controls, and saving the best model for evaluation.

The final test-set metrics are calculated in `03_evaluation.ipynb`.


## 1. Import Required Libraries

This stage imports the libraries used throughout the training pipeline.

- **PyTorch** provides tensors, neural-network layers, optimization, and data loading.
- **Torchvision** provides ImageNet-pretrained CNN architectures and image transforms.
- **PIL** reads the CUB-200 image files.
- **NumPy, Matplotlib, and JSON** support numerical processing, visualization, and experiment records.
- **Scikit-learn** creates a stratified training-validation split.

Keeping the imports in one cell makes the notebook easier to run from the beginning and makes the
software dependencies explicit.


In [ ]:
# Import the libraries used by the training pipeline.
from pathlib import Path
import json, random, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch, torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split


## 2. Define Training Configuration and Output Paths

This stage defines the experiment settings before any model is trained.

The classifier predicts **200 bird classes** and uses 224 x 224 RGB inputs, which is the standard
input size for the selected ImageNet-pretrained networks. The batch size, maximum number of epochs,
learning rates, weight decay, validation proportion, random seed, and early-stopping patience control
the optimization process.

Two learning rates are used because the notebook first trains the new classifier head and later
fine-tunes the pretrained backbone. The `models/checkpoints` folder stores the best weights, while
`models/history` stores loss, accuracy, and learning-rate histories for later analysis.


In [ ]:
# Training configuration and output locations.
NUM_CLASSES=200; IMAGE_SIZE=224; BATCH_SIZE=32; MAX_EPOCHS=35; WARMUP=5; LR1=1e-3; LR2=1e-4; WD=3e-4; PAT=5; VS=0.2; RS=42; NW=2
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
ROOT=Path('..').resolve(); DATA=ROOT/'data'; CK=ROOT/'models'/'checkpoints'; HS=ROOT/'models'/'history'


## 3. Prepare Reproducibility, Transforms, Dataset, and DataLoaders

This stage builds the complete input pipeline.

First, random seeds and deterministic cuDNN settings make repeated runs more comparable. Training
images receive random resized crops, horizontal flips, rotations, and color jitter. These random
transformations create varied examples and help reduce overfitting. Validation and test images use
only deterministic resize, center crop, tensor conversion, and ImageNet normalization so that their
measurements are stable.

The `CUB` dataset reads the annotation format `image_name class_label`, loads RGB images, and returns
an image tensor together with its integer class label. The training annotations are split with
stratification, so each bird class is represented proportionally in both training and validation
subsets. The test loader remains separate and is not used to tune the model.


In [ ]:
# Reproducibility, device selection, transforms, dataset parsing, and stratified loaders.
def seed(s=42):
 random.seed(s); np.random.seed(s); torch.manual_seed(s)
 if torch.cuda.is_available(): torch.cuda.manual_seed(s); torch.cuda.manual_seed_all(s)
 torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
def dev():
 return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def t_tr():
 # Random augmentation is applied only to training images.
 return transforms.Compose([transforms.RandomResizedCrop(IMAGE_SIZE,scale=(0.8,1.0)),transforms.RandomHorizontalFlip(),transforms.RandomRotation(15),transforms.ColorJitter(0.2,0.2,0.2,0.05),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
def t_ev():
 # Validation and test preprocessing must remain deterministic.
 return transforms.Compose([transforms.Resize(256),transforms.CenterCrop(IMAGE_SIZE),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
class CUB(Dataset):
 def __init__(self,idir,at,tf=None):
  self.idir=Path(idir); self.tf=tf; self.samples=[]
  with open(at,'r',encoding='utf-8') as f:
   for l in f:
    l=l.strip()
    if not l: continue
    p,y=l.split(); self.samples.append((self.idir/p,int(y)))
 def __len__(self): return len(self.samples)
 def __getitem__(self,i):
  p,y=self.samples[i]; im=Image.open(p).convert('RGB'); return (self.tf(im) if self.tf else im),y
def loaders(data_dir='data',bs=32,val=0.2,nw=2,rs=42):
 d=Path(data_dir); full=CUB(d/'Train',d/'train.txt'); lab=[y for _,y in full.samples]; idx=list(range(len(full)))
 # Stratification keeps all bird classes represented in train and validation subsets.
 a,b=train_test_split(idx,test_size=val,random_state=rs,stratify=lab)
 tr,va,te=CUB(d/'Train',d/'train.txt',t_tr()),CUB(d/'Train',d/'train.txt',t_ev()),CUB(d/'Test',d/'test.txt',t_ev())
 return DataLoader(Subset(tr,a),bs,True,num_workers=nw,pin_memory=torch.cuda.is_available()),DataLoader(Subset(va,b),bs,False,num_workers=nw,pin_memory=torch.cuda.is_available()),DataLoader(te,bs,False,num_workers=nw,pin_memory=torch.cuda.is_available())


## 4. Build the Transfer-Learning Model

This stage creates one of the supported ImageNet-pretrained architectures: ResNet18, ResNet50, or
EfficientNet-B0. ImageNet pretraining provides general visual features such as edges, textures, and
shapes, which is useful because CUB-200 has a relatively limited number of training images per class.

The original ImageNet classifier is replaced with a new linear layer containing 200 outputs. Training
then follows two stages:

1. **Classifier warm-up:** freeze the pretrained feature extractor and train only the new classifier.
2. **Full fine-tuning:** unfreeze the complete network and update it using a smaller learning rate.

This approach reduces the risk of destroying useful pretrained features and helps limit overfitting.


In [ ]:
# Build one of the supported ImageNet-pretrained architectures.
def model_of(name='resnet50',pre=True):
 n=name.lower()
 if n=='resnet18': m=models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pre else None); m.fc=nn.Linear(m.fc.in_features,NUM_CLASSES)
 elif n=='resnet50': m=models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pre else None); m.fc=nn.Linear(m.fc.in_features,NUM_CLASSES)
 elif n=='efficientnet_b0': m=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pre else None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,NUM_CLASSES)
 else: raise ValueError('unsupported')
 return m
def freeze(m):
 # Stage 1 trains only the new classifier to reduce overfitting.
 [setattr(p,'requires_grad',False) for p in m.parameters()]
 if hasattr(m,'fc'):
  [setattr(p,'requires_grad',True) for p in m.fc.parameters()]
 else:
  [setattr(p,'requires_grad',True) for p in m.classifier.parameters()]
def unfreeze(m):
 # Stage 2 fine-tunes the complete network with a smaller learning rate.
 [setattr(p,'requires_grad',True) for p in m.parameters()]


## 5. Define One-Epoch Training, Validation, and History Plots

The `run_epoch` function performs one pass over either the training or validation loader. It switches
the model between training and evaluation modes, computes cross-entropy loss, updates weights only in
training mode, and calculates top-1 accuracy.

Automatic mixed precision is enabled when a CUDA device is available. It can reduce GPU memory use
and speed up training while the gradient scaler helps preserve numerical stability. The `plot_hist`
function visualizes loss, accuracy, learning rate, and the best validation epoch. The dashed line
shows when the pipeline changes from classifier warm-up to full fine-tuning.


In [ ]:
# Run one training or validation epoch and plot the recorded history.
def run_epoch(m,ldr,crit,opt,device,scaler,train=True):
 m.train() if train else m.eval(); tl=0.; c=t=0
 ctx=torch.enable_grad() if train else torch.no_grad()
 with ctx:
  for x,y in tqdm(ldr,desc='train' if train else 'val',leave=False,dynamic_ncols=True):
   x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
   if train: opt.zero_grad(set_to_none=True)
   with torch.amp.autocast(device_type=device.type,enabled=device.type=='cuda'):
    o=m(x); loss=crit(o,y)
   if train:
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
   tl+=loss.item()*x.size(0); p=o.argmax(1); c+=(p==y).sum().item(); t+=y.size(0)
 return tl/t, c/t
def plot_hist(h):
 e=np.arange(1,len(h['tr_loss'])+1); fig,ax=plt.subplots(2,2,figsize=(14,10))
 # The dashed line marks the switch from classifier warm-up to fine-tuning.
 ax[0,0].plot(e,h['tr_loss'],label='train'); ax[0,0].plot(e,h['va_loss'],label='val'); ax[0,0].axvline(WARMUP+0.5,color='gray',ls='--'); ax[0,0].set_title('loss'); ax[0,0].legend()
 ax[0,1].plot(e,np.array(h['tr_acc'])*100,label='train'); ax[0,1].plot(e,np.array(h['va_acc'])*100,label='val'); ax[0,1].axvline(WARMUP+0.5,color='gray',ls='--'); ax[0,1].set_title('acc(%)'); ax[0,1].legend()
 ax[1,0].plot(e,h['lr']); ax[1,0].set_title('learning rate')
 be=int(np.argmax(h['va_acc']))+1; ax[1,1].bar(['best val%','best epoch','epochs'],[float(np.max(h['va_acc'])*100),be,len(e)]); ax[1,1].set_title('training status')
 plt.tight_layout(); plt.show()


## 6. Train, Validate, and Save the Best Checkpoint

This stage contains the main experiment loop. The model uses cross-entropy loss with label smoothing,
which discourages over-confident predictions and can improve generalization. AdamW provides adaptive
optimization with decoupled weight decay for regularization.

The validation accuracy controls a `ReduceLROnPlateau` scheduler. When validation performance stops
improving, the learning rate is reduced. After the warm-up epochs, all layers are unfrozen and a new
optimizer is created with the smaller fine-tuning learning rate.

Only the checkpoint with the best validation accuracy is saved. Early stopping terminates training
when validation accuracy has failed to improve for the configured patience period. The complete
training history is written as JSON so that the results can be reproduced and discussed later.


In [ ]:
# Train, validate, save the best checkpoint, and write the history JSON.
def train_model(name='resnet50'):
 seed(RS); device=dev(); tr,va,_=loaders(DATA,BATCH_SIZE,VS,NW,RS); m=model_of(name,True).to(device); crit=nn.CrossEntropyLoss(label_smoothing=0.1)
 freeze(m); opt=AdamW(filter(lambda p:p.requires_grad,m.parameters()),lr=LR1,weight_decay=WD); sch=ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=2); sc=torch.amp.GradScaler(device=device.type,enabled=device.type=='cuda')
 CK.mkdir(parents=True,exist_ok=True); HS.mkdir(parents=True,exist_ok=True); cp=CK/f'{name}_best.pth'; hp=HS/f'{name}_history.json'
 h={'tr_loss':[],'tr_acc':[],'va_loss':[],'va_acc':[],'lr':[]}; best=-1.; bad=0
 for ep in range(1,MAX_EPOCHS+1):
  if ep==WARMUP+1:
   # Replace the optimizer when the full backbone becomes trainable.
   unfreeze(m); opt=AdamW(m.parameters(),lr=LR2,weight_decay=WD); sch=ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=2)
  tl,ta=run_epoch(m,tr,crit,opt,device,sc,True); vl,va_acc=run_epoch(m,va,crit,opt,device,sc,False); sch.step(va_acc); lr=opt.param_groups[0]['lr']
  h['tr_loss'].append(tl); h['tr_acc'].append(ta); h['va_loss'].append(vl); h['va_acc'].append(va_acc); h['lr'].append(lr)
  print(f'ep{ep:02d} train={ta*100:.2f}% val={va_acc*100:.2f}% lr={lr:.6f}')
  if va_acc>best:
   # Keep the checkpoint with the best validation accuracy.
   best=va_acc; bad=0; torch.save({'epoch':ep,'model_name':name,'validation_accuracy':va_acc,'model_state_dict':m.state_dict(),'optimizer_state_dict':opt.state_dict(),'scheduler_state_dict':sch.state_dict()},cp)
  else: bad+=1
  if bad>=PAT: break
 with open(hp,'w',encoding='utf-8') as f: json.dump(h,f,indent=4)
 print('best val:',f'{best*100:.2f}%','\ncheckpoint:',cp,'\nhistory:',hp); plot_hist(h); return h


## 7. Select and Run a Training Experiment

Choose the architecture in `MODEL`. The available options are `resnet18`, `resnet50`, and
`efficientnet_b0`. The call to `train_model` is intentionally commented out so that the notebook does
not start a long training job accidentally when all cells are run.

After selecting a model, uncomment the training line. The notebook will print epoch-level training and
validation results, display the learning curves, save the best checkpoint, and save the history JSON.
The checkpoint generated here is the input used by `03_evaluation.ipynb`.


In [ ]:
# Choose one architecture. Uncomment the next line when ready to train.
MODEL='resnet50'  # resnet18 | resnet50 | efficientnet_b0
# hist=train_model(MODEL)  # Start training and generate the charts.


# Training Summary

The training pipeline uses transfer learning, augmentation, label smoothing, weight decay, a
stratified validation split, learning-rate scheduling, early stopping, and best-checkpoint selection.
Together, these choices provide regularization and monitoring against overfitting while allowing the
pretrained visual features to adapt to the 200 CUB-200 bird species.

The next stage is to evaluate the selected checkpoint on the untouched official test set.
